# Selecting Scorio GPQA answers with `scorio.aggregate`

This notebook selects one answer per question for ten questions from a single field.
Answer and verifier columns are projected from the Bucket. A final section reads one
complete low-reasoning pool to reproduce confidence signals from the top-20 distributions.


In [1]:
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from IPython.display import display

from scorio import agg, eval

BUCKET_ROOT = "hf://buckets/harimo/scorio-gpqa"


def pool_path(model, question_id):
    return f"{BUCKET_ROOT}/data/{model}/super_gpqa/q{question_id:04d}.parquet"


def read_pools(model, question_ids, columns, max_workers=2):
    """Read selected columns from question files, preserving question order."""
    paths = [pool_path(model, q) for q in question_ids]

    def read_one(path):
        return pq.read_table(path, columns=columns)

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        tables = list(executor.map(read_one, paths))
    return pa.concat_tables(tables).to_pandas()

model_name = "gpt-oss-20b_low"
field_start = 0
question_count = 10
question_ids = range(field_start, field_start + question_count)
columns = [
    "full_data_id", "seed", "field", "evalscope_extracted_answer", "evalscope_is_correct",
    "cv3b_abc_A", "llmv_problem_understanding_expected",
    "llmv_reasoning_validity_expected", "llmv_conclusion_support_expected",
]

rows = read_pools(model_name, question_ids, columns).sort_values(["full_data_id", "seed"])
M, N = question_count, 80

answers = rows.evalscope_extracted_answer.to_numpy().reshape(M, N).astype(object)
answers[answers == "NotFound"] = None
cv3b = rows.cv3b_abc_A.to_numpy().reshape(M, N)

llmv_expected = (
    rows.llmv_problem_understanding_expected
    + rows.llmv_reasoning_validity_expected
    + rows.llmv_conclusion_support_expected
).to_numpy().reshape(M, N) / 3
llmv = (llmv_expected - 1) / 19

accepted = [
    set(group.loc[group.evalscope_is_correct.astype(bool), "evalscope_extracted_answer"])
    for _, group in rows.groupby("full_data_id", sort=True)
]

print("field:", rows.field.iloc[0])
print(answers.shape, cv3b.shape, llmv.shape)


/tmp/scorio_uv_cache/archive-v0/qlDst6OSBXQ1PndVk4Wef/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


field: Aeronautical and Astronautical Science and Technology
(10, 80) (10, 80) (10, 80)


## One question and eight candidates


In [2]:
question_offset = 0
pool = answers[question_offset, :8]
scores = cv3b[question_offset, :8]

print("candidates    ", list(pool))
print("P(correct)    ", scores.round(3))
print("first sample  ", pool[0])
print("majority_vote ", agg.majority_vote(pool))
print("best_of_n     ", agg.best_of_n(pool, scores))
print("accepted      ", accepted[question_offset])


candidates     ['D', 'F', 'D', 'F', 'D', 'F', 'D', 'F']
P(correct)     [0.    0.998 0.    1.    0.    0.997 0.001 0.999]
first sample   D
majority_vote  D
best_of_n      F
accepted       {'F'}


## Select one answer for every question


In [3]:
def accuracy(selected):
    hit = np.array([[int(answer in accepted[i])] for i, answer in enumerate(selected)])
    mu, sigma = eval.avg(hit)
    return {"accuracy": round(float(mu), 3), "sigma": round(float(sigma), 3)}


n = 8
A, V, L = answers[:, :n], cv3b[:, :n], llmv[:, :n]
results = {
    "first sample": accuracy(A[:, 0]),
    "majority_vote": accuracy(agg.majority_vote(A)),
    "weighted_majority_vote (CV3B)": accuracy(agg.weighted_majority_vote(A, V)),
    "best_of_n (CV3B)": accuracy(agg.best_of_n(A, V)),
    "best_of_n (reference-free verifier)": accuracy(agg.best_of_n(A, L)),
}
display(pd.DataFrame(results).T.sort_values("accuracy", ascending=False))


,accuracy,sigma
weighted_majority_vote (CV3B),0.6,0.224
best_of_n (CV3B),0.6,0.224
best_of_n (reference-free verifier),0.3,0.224
first sample,0.1,0.224
majority_vote,0.1,0.224


## Sample-budget sweep


In [4]:
budgets = [1, 2, 4, 8, 16, 32, 80]
sweep = pd.DataFrame({
    n: {
        "majority_vote": accuracy(agg.majority_vote(answers[:, :n]))["accuracy"],
        "weighted (CV3B)": accuracy(
            agg.weighted_majority_vote(answers[:, :n], cv3b[:, :n])
        )["accuracy"],
        "best_of_n (CV3B)": accuracy(
            agg.best_of_n(answers[:, :n], cv3b[:, :n])
        )["accuracy"],
    }
    for n in budgets
}).T
sweep.index.name = "samples"
display(sweep)


,majority_vote,weighted (CV3B),best_of_n (CV3B)
samples,,,
1,0.1,0.1,0.1
2,0.1,0.2,0.2
4,0.1,0.4,0.4
8,0.1,0.6,0.6
16,0.2,0.8,0.8
32,0.3,0.9,0.9
80,0.3,1.0,1.0


## Confidence from the top-20 distributions


In [5]:
full_pool = pq.read_table(pool_path(model_name, field_start)).to_pylist()


def topk_logprobs(record):
    return [[item["logprob"] for item in position]
            for position in record["tokens"]["completion_topk_logprobs_list"]]


signals = []
for record in full_pool[:3]:
    topk = topk_logprobs(record)
    signals.append({
        "seed": record["seed"],
        "self_certainty": agg.self_certainty(topk),
        "deepconf": agg.deepconf_confidence(topk),
        "negative_entropy": -agg.token_entropy(topk),
        "max_probability": agg.max_softmax_probability(topk),
        "logprob_margin": agg.logprob_margin(topk),
    })
display(pd.DataFrame(signals).round(4))


,seed,self_certainty,deepconf,negative_entropy,max_probability,logprob_margin
0,0,7.3558,10.3565,-0.6241,0.7878,5.0020
1,1,9.6287,12.6257,-0.3480,0.8831,7.1690
2,2,7.4636,10.4630,-0.5527,0.8129,5.0492


The [aggregation reference](https://github.com/mohsenhariri/scorio/blob/main/scorio/aggregate/README.md)
maps published methods to their confidence signal and selection rule.
